### 04 — Clinical Feature Extraction using ClinicalBERT

What this notebook does:

- load the existing training, validation, and test splits
- inspect the clinical variables used by the model
- convert the structured clinical data into short natural-language text
- load the pretrained `Bio_ClinicalBERT` tokenizer and model
- test the tokenisation process
- extract contextual clinical token embeddings
- use the reusable `ClinicalBERTEncoder` class from `src/clinical_encoder.py`
- freeze the pretrained ClinicalBERT parameters
- test ClinicalBERT on a batch of clinical text
- save the generated `clinical_text` column back into the dataset splits.

The output from this notebook is a sequence of 768-dimensional clinical token features that will later be combined with the EfficientNet visual features during multimodal fusion.


In [1]:
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn

from transformers import AutoTokenizer, AutoModel

c:\Users\Akash\Desktop\Python\breast-ultrasound-fyp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Define Project Paths

The notebook uses the existing project folders for the raw ultrasound images, processed dataset splits, and trained model checkpoints.

In [2]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

print(len(train_df), len(val_df), len(test_df))

176 38 38


### 2. Inspect the Clinical Data

The clinical branch uses four structured variables:

- Age
- Tissue composition
- Signs
- Symptoms

These variables describe patient-level clinical information and are used as the input to ClinicalBERT.


In [3]:
clinical_columns = [
    "Age",
    "Tissue_composition",
    "Signs",
    "Symptoms"
]

display(train_df[clinical_columns].head(10))

,Age,Tissue_composition,Signs,Symptoms
0,63,not available,not available,not available
1,39,heterogeneous: predominantly fat,no,breast injury
2,42,heterogeneous: predominantly fibroglandular,palpable,family history of breast/ovarian cancer
3,not available,heterogeneous: predominantly fat,no,no
4,47,homogeneous: fibroglandular,no,not available
5,not available,not available,no,no
6,57,heterogeneous: predominantly fat,no,not available
7,51,not available,not available,not available
8,35,not available,not available,not available
9,58,heterogeneous: predominantly fat,no,no


Inspect the value counts for each variable and identify entries such as 'not available' that are stored as text in the dataset.


In [4]:
for column in clinical_columns:
    print(f"\n--- {column} ---")
    print(train_df[column].value_counts(dropna=False).head(20))


--- Age ---
Age
not available    27
57                8
43                6
63                5
62                5
47                4
51                4
35                4
48                4
50                4
68                4
66                4
45                4
65                4
69                4
76                4
60                4
56                4
39                3
42                3
Name: count, dtype: int64

--- Tissue_composition ---
Tissue_composition
heterogeneous: predominantly fat                         57
heterogeneous: predominantly fibroglandular              39
not available                                            34
homogeneous: fibroglandular                              28
homogeneous: fat                                         15
lactating&homogeneous: fibroglandular                     2
lactating&heterogeneous: predominantly fibroglandular     1
Name: count, dtype: int64

--- Signs ---
Signs
no                          101
palpable   

### 3. Convert Structured Clinical Data into Text

ClinicalBERT expects text rather than separate spreadsheet columns.

The four clinical variables are combined into a short sentence using a template

Using the same format for every case gives ClinicalBERT a consistent representation of the structured clinical information.


In [5]:
def create_clinical_text(row):
    age = str(row["Age"]).strip()
    tissue = str(row["Tissue_composition"]).strip()
    signs = str(row["Signs"]).strip()
    symptoms = str(row["Symptoms"]).strip()

    text = (
        f"Patient age: {age} years. "
        f"Tissue composition: {tissue}. "
        f"Clinical signs: {signs}. "
        f"Symptoms: {symptoms}."
    )

    return text

The text representation is created for the training, validation, and test sets and stored in a new clinical_text column.


In [6]:
train_df["clinical_text"] = train_df.apply(
    create_clinical_text,
    axis=1
)

val_df["clinical_text"] = val_df.apply(
    create_clinical_text,
    axis=1
)

test_df["clinical_text"] = test_df.apply(
    create_clinical_text,
    axis=1
)

In [7]:
for text in train_df["clinical_text"].head(5):
    print(text)
    print()

Patient age: 63 years. Tissue composition: not available. Clinical signs: not available. Symptoms: not available.

Patient age: 39 years. Tissue composition: heterogeneous: predominantly fat. Clinical signs: no. Symptoms: breast injury.

Patient age: 42 years. Tissue composition: heterogeneous: predominantly fibroglandular. Clinical signs: palpable. Symptoms: family history of breast/ovarian cancer.

Patient age: not available years. Tissue composition: heterogeneous: predominantly fat. Clinical signs: no. Symptoms: no.

Patient age: 47 years. Tissue composition: homogeneous: fibroglandular. Clinical signs: no. Symptoms: not available.



### 4. Load ClinicalBERT

The clinical text encoder uses `emilyalsentzer/Bio_ClinicalBERT`.

Both the pretrained tokenizer and pretrained model are loaded from Hugging Face.

ClinicalBERT produces contextual token representations that can capture relationships between the different clinical terms in the input text.


In [8]:
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

clinical_bert = AutoModel.from_pretrained(
    MODEL_NAME
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16211.21it/s]
[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The model is moved to the GPU when CUDA is available; otherwise, it runs on the CPU.


In [9]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

clinical_bert = clinical_bert.to(device)

print("Device:", device)

Device: cuda


### 5. Test the Tokenisation Process

The tokenizer converts the text into model inputs including:

- `input_ids` — token identifiers
- `attention_mask` — indicates which positions contain real tokens rather than padding.

In [10]:
sample_text = train_df.iloc[0]["clinical_text"]

print(sample_text)

Patient age: 63 years. Tissue composition: not available. Clinical signs: not available. Symptoms: not available.


In [11]:
encoded = tokenizer(
    sample_text,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=64
)

print(encoded.keys())
print(encoded["input_ids"].shape)
print(encoded["attention_mask"].shape)

KeysView({'input_ids': tensor([[ 101, 5351, 1425,  131, 5519, 1201,  119, 7918, 5239,  131, 1136, 1907,
          119, 7300, 5300,  131, 1136, 1907,  119, 8006,  131, 1136, 1907,  119,
          102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1]])})
torch.Size([1, 25])
torch.Size([1, 25])


### 6. Extract ClinicalBERT Features

The tokenised inputs are moved to the selected device and passed through ClinicalBERT in evaluation mode.

torch.no_grad() is used because this step is only testing feature extraction and does not require gradient calculations.


In [12]:
encoded = {
    key: value.to(device)
    for key, value in encoded.items()
}

In [13]:
clinical_bert.eval()

with torch.no_grad():
    output = clinical_bert(**encoded)

In [14]:
print(output.last_hidden_state.shape)

torch.Size([1, 25, 768])


### 7. Use the Reusable ClinicalBERT Encoder

The `ClinicalBERTEncoder` class from `src/clinical_encoder.py` is imported.


In [15]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.clinical_encoder import ClinicalBERTEncoder

In [16]:
clinical_encoder = ClinicalBERTEncoder().to(device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19893.85it/s]
[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 8. Freeze the ClinicalBERT Parameters

The pretrained ClinicalBERT weights are frozen before multimodal training.


In [17]:
for parameter in clinical_encoder.bert.parameters():
    parameter.requires_grad = False

In [18]:
print(
    sum(
        p.numel()
        for p in clinical_encoder.parameters()
        if p.requires_grad
    )
)

0


### 9. Test ClinicalBERT on a Batch

A batch of eight clinical-text samples is tokenised together.

In [19]:
texts = train_df["clinical_text"].iloc[:8].tolist()

batch = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

The tokenised batch is moved to the same device as the ClinicalBERT encoder.


In [20]:
batch = {
    key: value.to(device)
    for key, value in batch.items()
}

The batch is passed through the clinical encoder.

The resulting tensor contains one 768-dimensional feature vector for every token in every clinical-text sequence.


In [21]:
clinical_encoder.eval()

with torch.no_grad():
    clinical_tokens = clinical_encoder(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch.get("token_type_ids")
    )

print(clinical_tokens.shape)

torch.Size([8, 42, 768])


### 10. Save the Clinical Text into the Dataset Splits

The updated training, validation, and test DataFrames are saved back to the processed CSV files.


In [22]:
train_df.to_csv(
    PROCESSED_DIR / "train.csv",
    index=False
)

val_df.to_csv(
    PROCESSED_DIR / "val.csv",
    index=False
)

test_df.to_csv(
    PROCESSED_DIR / "test.csv",
    index=False
)